# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment (Python 3.8 or newer). It uses [Skyfield](https://rhodesmill.org/skyfield/), a standard Python library for satellite positions, and the helper file `Satellites/sat_tools.py` next to this notebook. The orbits of all satellites are stored in `Satellites/tle/`, a snapshot from [CelesTrak](https://celestrak.org/), so the notebook needs no internet after the install.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install skyfield numpy matplotlib --quiet

**Imports**

In [ ]:
# --- Imports: libraries and functions used in this notebook ---
import numpy as np
import matplotlib.pyplot as plt

from Satellites.sat_tools import (SNAPSHOT_DAY, constellations, orbit_basics,
                                  customer_for, serve_customer,
                                  walker_shell, evaluate_shell, beam_capacity, DISH_BEAM, PHONE_BEAM,
                                  masts_needed, bridge_figure, next_passes)

### 1. Introduction

Thousands of satellites orbit the Earth for communication. This notebook uses their real orbits to answer two questions: which systems can serve a customer far away from any mast (sections 2 and 3), and where satellites serve better than masts (section 4). The navigation satellites are covered in `Network_GNSS.ipynb`.

The orbit of every satellite is published as a two-line element set (TLE). It describes the size, shape and orientation of the orbit and the position of the satellite on it, valid around one moment, the epoch. The SGP4 model computes the position of the satellite at any time from it. It is accurate to about a kilometre near the epoch and slowly gets worse over weeks, which is accurate enough to know where a satellite is in the sky.

The snapshot in `Satellites/tle/` holds all active satellites of these systems:

| System | Use | Orbit |
|---|---|---|
| Starlink | broadband to dishes, and to phones | low Earth orbit, about 470 km, in shells inclined 43°, 53°, 70° and 97.6° |
| OneWeb | broadband to dishes | about 1,200 km, inclined 88° |
| Iridium | voice and data to handheld phones | about 780 km, inclined 86° |
| geostationary satellites | TV, broadband, ships | 35,786 km above the equator |

**Declarations**

Your customer in section 2 depends on your matriculation number.

In [ ]:
# --- Declarations: editable variables for students ---
MATRICULATION = "12-345-678"     # <-- Change this. Your matriculation number

print(f"Orbit snapshot: {SNAPSHOT_DAY:%Y-%m-%d}")
for label, s in constellations().items():
    print(f"   {label:15s}{len(s['satrecs']):6d} satellites ({s['note']})")

#### 1.1 Orbits in Numbers

The period of a circular orbit follows from Kepler's third law:

$$
T = 2\pi\sqrt{\frac{a^3}{\mu}},
$$

where
- $a$: distance of the satellite from the centre of the Earth (km),
- $\mu = 398{,}600\ \text{km}^3/\text{s}^2$: gravitational parameter of the Earth.

The higher the orbit, the slower the satellite and the larger the area from which it is visible, its footprint (here above 25° elevation). At 35,786 km, the period equals one day, so the satellite stays above one point of the equator.

In [ ]:
for name, h in (("Starlink", 470), ("Iridium", 780), ("OneWeb", 1200), ("geostationary", 35786)):
    o = orbit_basics(h)
    print(f"{name:14s}{h:7,d} km: period {o['period_min']:7.1f} min, speed {o['speed_kms']:4.1f} km/s, "
          f"footprint radius at 25° {o['footprint_radius_km']:6,.0f} km, delay straight down {h / 299.792:6.1f} ms")

### 2. Serving a Remote Customer

You now plan for a customer far away from any mast. Each customer has a place, a need, and a terrain mask: mountains or valley walls that hide the lowest part of the sky. The question is which real systems can serve the customer, how well, and why.

Whether a system covers a place depends on three things:
- Inclination: a satellite never flies over latitudes higher than the inclination of its orbit, and it spends most of its time near that latitude. A shell therefore serves latitudes up to about its inclination plus its footprint, which is a few degrees in a low orbit. Shells close to 90° pass over every place.
- Elevation: a dish or a phone needs the satellite above a minimum elevation (here 25° for broadband dishes, 8° for Iridium and 10° for a geostationary dish), and above the terrain.
- Geostationary satellites stay above the equator. From high latitudes they appear low above the horizon, and beyond about 81° they are below it.

Two more numbers matter for the link. The delay is the distance divided by the speed of light. An internet connection goes from the user to the satellite, down to a gateway and back, so a round trip takes about four times the one-way delay in the table. The Doppler shift comes from the speed of the satellite: a low satellite moves at 7.6 km/s, so its carrier shifts by up to about 40 kHz at 2 GHz while it passes, and about six times more in the Ku band (about 12 GHz) of dish services. An LTE or 5G subcarrier is only 15 kHz wide, so phones that connect to satellites directly (3GPP non-terrestrial networks) correct their frequency and timing in advance, using their own GNSS position and the orbit of the satellite.

The table and the plot cover one simulated day in steps of one minute. The last line names the systems that cover your customer: available at least 95 % of the day, with no gap longer than your customer accepts. Whether the terminal and the data rate fit the need is for you to judge.

In [ ]:
customer = customer_for(MATRICULATION)
result = serve_customer(customer)

### 3. A What-If Shell

Could fewer satellites serve your customer? A Walker constellation places `planes` orbital planes evenly around the Earth, each with `per_plane` satellites, all at the same altitude and inclination. The main trade-offs:
- An inclination close to the latitude of your customer puts the satellites where they spend most of their time. If it is too low, they never come close enough.
- A higher altitude gives a larger footprint, so fewer satellites cover the same area, but the delay grows and the link gets weaker.
- Fewer planes leave longer gaps between passes, and fewer satellites per plane leave gaps along each plane.

Each run takes a few seconds, and the last line says whether your shell covers your customer.

In [ ]:
SHELL = dict(alt_km=550, inc_deg=53, planes=24, per_plane=20)     # <-- Change this. Your shell
MIN_EL = 25                                                        # elevation a dish needs (the terrain mask is added)

shell = walker_shell(SHELL["alt_km"], SHELL["inc_deg"], SHELL["planes"], SHELL["per_plane"])
mine = evaluate_shell(customer, shell, min_el=MIN_EL, label=f"{len(shell)} sats, {SHELL['inc_deg']}°")

### 4. Masts or Satellites?

A satellite serves many spot beams, and the beams reuse the frequencies like cells: with a reuse factor of about 4, neighbouring beams use different frequencies. The width of a beam is about $1.02\,\lambda/D$ for an antenna of diameter $D$. The area of a beam on the ground is therefore fixed by the antenna and the altitude, and so is its capacity per km². Two examples:
- a broadband service to dishes: Ku band at 11.7 GHz, a 1 m antenna on the satellite, 2 GHz of spectrum shared by 4 beams, 3 b/s/Hz;
- a service directly to phones: 1.9 GHz, a 5 m antenna on the satellite, 5 MHz of spectrum, 1 b/s/Hz, because the small antenna of a phone allows little more.

A constellation adds capacity only by adding satellites. A few hundred satellites are enough to cover the latitudes where people live, but each carries only tens of Gb/s, spread over a large footprint. Coverage therefore takes hundreds of satellites, and capacity takes thousands, just as a city needs many more masts for capacity than for coverage (section 6 of `Network_Cellular.ipynb`).

On the ground, a mast of 30 m reaches at most its radio horizon, about 28 km for a phone at 1.5 m, and much less in hills. Below the knee of the cellular network, masts carry far less than they could: they are built for coverage, not for capacity.

The figure below puts both on one picture, with the cellular model of `Network_Cellular.ipynb`.

In [ ]:
for name, b in (("dish beam", DISH_BEAM), ("direct-to-phone beam", PHONE_BEAM)):
    r = beam_capacity(**b)
    print(f"{name:22s}: {r['beam_area_km2']:6.0f} km2, {r['beam_mbps']:7.1f} Mb/s per beam, {r['mbps_per_km2']:.3f} Mb/s per km2")
m = masts_needed(customer)
print(f"masts of 30 m for your customer: reach at most {m['reach_km']:.0f} km each -> at least {m['masts']} mast(s)")
print(f"   {m['note']}\n" if m["note"] else "")
bridge = bridge_figure(customer)

### 5. Optional: Look Up

The International Space Station passes over Switzerland several times a day. Around dawn and dusk, it shines brightly in the sunlight while the sky is already dark. The cell below lists the passes near the snapshot date; for other dates, check a site such as heavens-above.com, since the orbit changes.

In [ ]:
LAT, LON = 47.3763, 8.5480       # <-- Change this. Your place (degrees)

next_passes(LAT, LON, "ISS (ZARYA)", days=3)